In [0]:
# Instalación de librerías necesarias con versiones fijas para reproducibilidad
%pip install mlflow scikit-learn==1.4.0 pandas==2.1.4 numpy==1.26.3 matplotlib==3.8.2 seaborn==0.13.1 ydata-profiling==4.8.3 optuna==3.5.0 category-encoders==2.6.3 xgboost==2.0.3 --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Reiniciar el kernel de Python para cargar las librerías instaladas
dbutils.library.restartPython()

In [0]:
# Configurar nombre del experimento MLflow
EXPERIMENT_NAME = "/Users/{}/riesgo_ml_tradicional".format(
    spark.sql("SELECT current_user()").collect()[0][0]
)

print("=" * 70)
print("🔬 CONFIGURACIÓN DE EXPERIMENTO MLFLOW")
print("=" * 70)
print(f"\n📊 Experimento: {EXPERIMENT_NAME}")
print("\nEste notebook carga el mejor modelo entrenado en el notebook anterior.")
print("=" * 70)

🔬 CONFIGURACIÓN DE EXPERIMENTO MLFLOW

📊 Experimento: /Users/dan.pechi@databricks.com/riesgo_ml_tradicional

Este notebook carga el mejor modelo entrenado en el notebook anterior.


In [0]:
# Importar librerías necesarias
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("✅ Librerías importadas exitosamente")

✅ Librerías importadas exitosamente


In [0]:
# Buscar el mejor modelo en el experimento MLflow
print("=" * 70)
print("🔍 BUSCANDO MEJOR MODELO EN EXPERIMENTO MLFLOW")
print("=" * 70)

client = MlflowClient()

# Obtener el experimento
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(f"❌ Experimento '{EXPERIMENT_NAME}' no encontrado. Asegúrate de haber ejecutado el notebook de entrenamiento primero.")

print(f"\n✅ Experimento encontrado: {EXPERIMENT_NAME}")
print(f"   Experiment ID: {experiment.experiment_id}")

# Buscar todos los runs del experimento ordenados por F1-score
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="",
    order_by=["metrics.test_f1_score DESC"],
    max_results=1
)

if not runs:
    raise ValueError(f"❌ No se encontraron runs en el experimento '{EXPERIMENT_NAME}'")


# Seleccionar el mejor modelo
best_run = runs[0]
final_run_id = best_run.info.run_id
best_run_name = best_run.data.tags.get('mlflow.runName', 'Sin nombre')
best_f1_score = best_run.data.metrics.get('test_f1_score', 0)

print("=" * 70)
print(f"✅ MEJOR MODELO SELECCIONADO: {best_run_name}")
print(f"   Run ID: {final_run_id}")
print(f"   F1-Score: {best_f1_score:.4f}")
print("=" * 70)

# Cargar el modelo desde MLflow
print("\n🔄 Cargando modelo desde MLflow...\n")

model_uri = f"runs:/{final_run_id}/model"
final_model = mlflow.xgboost.load_model(model_uri)

🔍 BUSCANDO MEJOR MODELO EN EXPERIMENTO MLFLOW

✅ Experimento encontrado: /Users/dan.pechi@databricks.com/riesgo_ml_tradicional
   Experiment ID: 2106771804875176
✅ MEJOR MODELO SELECCIONADO: XGBoost_Optimized_Final
   Run ID: 3f29aaa2cefc4fdfa77eac41dfb5b4e1
   F1-Score: 0.8473

🔄 Cargando modelo desde MLflow...



## 📚 MLflow Model Registry - Gestión de Modelos en Producción

MLflow Model Registry es un repositorio centralizado para gestionar el ciclo de vida completo de modelos ML.

**Funcionalidades clave:**

* **Registro de Modelos**: Almacenar modelos con metadata
* **Versionado**: Mantener historial de versiones
* **Stage Management**: Transiciones entre etapas (None → Staging → Production → Archived)
* **Anotaciones**: Descripciones y tags para documentación
* **Lineage**: Trazabilidad desde datos hasta modelo
* **Control de Acceso**: Permisos y gobernanza

**Etapas del ciclo de vida:**

1. **None**: Modelo recién registrado, sin etapa asignada
2. **Staging**: Modelo en pruebas pre-producción
3. **Production**: Modelo activo sirviendo predicciones
4. **Archived**: Modelo retirado, mantenido para historial

**💡 Mejores prácticas:**

* Usar nombres descriptivos para modelos
* Documentar cada versión con descripciones claras
* Implementar proceso de aprobación antes de Production
* Mantener al menos una versión en Production
* Archivar versiones obsoletas

In [0]:
# Registrar el modelo final en MLflow Model Registry
print("=" * 70)
print("📚 REGISTRO EN MLFLOW MODEL REGISTRY")
print("=" * 70)

# Definir nombre del modelo
model_name = "modelo_riesgo_fiscal_mx"

print(f"\n🔄 Registrando modelo '{model_name}'...")

# Registrar el modelo desde el run
model_uri = f"runs:/{final_run_id}/model"

try:
    # Registrar modelo
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )
    
    print(f"\n✅ Modelo registrado exitosamente")
    print(f"\n📊 Detalles del registro:")
    print(f"   • Nombre del modelo: {model_name}")
    print(f"   • Versión: {model_version.version}")
    print(f"   • Run ID: {final_run_id}")
    print(f"   • Estado actual: {model_version.current_stage}")
    
    # Guardar versión para uso posterior
    registered_model_version = model_version.version
    
except Exception as e:
    print(f"\n⚠️ Nota: {str(e)}")
    print("\nSi el modelo ya existe, se creará una nueva versión automáticamente.")
    
    # Obtener la última versión
    from mlflow.tracking import MlflowClient
    client = MlflowClient()
    
    # Intentar obtener versiones existentes
    try:
        versions = client.search_model_versions(f"name='{model_name}'")
        if versions:
            latest_version = max([int(v.version) for v in versions])
            registered_model_version = latest_version
            print(f"\n✅ Usando versión existente: {registered_model_version}")
        else:
            registered_model_version = 1
    except:
        registered_model_version = 1

print("\n" + "=" * 70)

📚 REGISTRO EN MLFLOW MODEL REGISTRY

🔄 Registrando modelo 'modelo_riesgo_fiscal_mx'...


Successfully registered model 'ai_specialist.default.modelo_riesgo_fiscal_mx'.
2026/01/28 06:07:22 WARNING mlflow.tracking._model_registry.fluent: Run with id 3f29aaa2cefc4fdfa77eac41dfb5b4e1 has no artifacts at artifact path 'model', registering model based on models:/m-565e2afce7bc40399545c234ed58bec0 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]


✅ Modelo registrado exitosamente

📊 Detalles del registro:
   • Nombre del modelo: modelo_riesgo_fiscal_mx
   • Versión: 1
   • Run ID: 3f29aaa2cefc4fdfa77eac41dfb5b4e1
   • Estado actual: None



🔗 Created version '1' of model 'ai_specialist.default.modelo_riesgo_fiscal_mx': https://dbc-9dcd6158-e299.cloud.databricks.com/explore/data/models/ai_specialist/default/modelo_riesgo_fiscal_mx/version/1?o=2226288096546970


In [0]:
# Añadir descripción y metadata al modelo registrado
print("=" * 70)
print("📝 AÑADIENDO DESCRIPCIÓN Y METADATA")
print("=" * 70)

from mlflow.tracking import MlflowClient
client = MlflowClient()

print(f"\n🔄 Actualizando metadata del modelo...")

# Descripción del modelo
model_description = """
Modelo de Clasificación de Riesgo Fiscal para la Autoridad Fiscal Federal de México.

**Algoritmo**: XGBoost Classifier optimizado con Optuna

**Objetivo**: Clasificar contribuyentes en tres niveles de riesgo:
- Alto: Requiere auditoría inmediata
- Medio: Monitoreo reforzado
- Bajo: Monitoreo estándar

**Features**: 
- Variables financieras (ingresos, deducciones)
- Historial de cumplimiento (auditorías, irregularidades, pagos tardíos)
- Características empresariales (sector, tamaño, antigüedad)

**Métricas de Rendimiento**:
- Test Accuracy: ~85-90%
- Test F1-Score: ~85-90%
- Optimizado para balance entre Precision y Recall

**Preprocesamiento**:
- Imputación de valores faltantes
- One-hot encoding de variables categóricas
- Estandarización de features numéricas

**Fecha de entrenamiento**: Enero 2026
**Dataset**: 5,000 registros sintéticos
**Entrenado por**: Equipo de Data Science - Autoridad Fiscal MX
"""

try:
    # Actualizar descripción del modelo registrado
    client.update_registered_model(
        name=model_name,
        description=model_description
    )
    
    # Añadir descripción específica de la versión
    version_description = f"""
    Versión {registered_model_version} - Modelo optimizado con Optuna (50 trials).
    
    Hiperparámetros optimizados para maximizar F1-score.
    Incluye pipeline completo de preprocesamiento.
    Listo para despliegue en ambiente de Staging.
    """
    
    client.update_model_version(
        name=model_name,
        version=registered_model_version,
        description=version_description
    )
    
    # Añadir tags
    client.set_model_version_tag(
        name=model_name,
        version=registered_model_version,
        key="task",
        value="classification"
    )
    
    client.set_model_version_tag(
        name=model_name,
        version=registered_model_version,
        key="algorithm",
        value="xgboost"
    )
    
    client.set_model_version_tag(
        name=model_name,
        version=registered_model_version,
        key="optimization",
        value="optuna"
    )
    
    client.set_model_version_tag(
        name=model_name,
        version=registered_model_version,
        key="use_case",
        value="riesgo_fiscal"
    )
    
    print("\n✅ Descripción y metadata actualizadas exitosamente")
    print("\n🏷️ Tags añadidos:")
    print("   • task: classification")
    print("   • algorithm: xgboost")
    print("   • optimization: optuna")
    print("   • use_case: riesgo_fiscal")
    
except Exception as e:
    print(f"\n⚠️ Error al actualizar metadata: {str(e)}")

print("\n" + "=" * 70)

📝 AÑADIENDO DESCRIPCIÓN Y METADATA

🔄 Actualizando metadata del modelo...

✅ Descripción y metadata actualizadas exitosamente

🏷️ Tags añadidos:
   • task: classification
   • algorithm: xgboost
   • optimization: optuna
   • use_case: riesgo_fiscal



### 📝 Resumen de MLflow Model Registry

**Acciones completadas:**

1. ✅ **Registro del modelo** en MLflow Model Registry
2. ✅ **Documentación** con descripciones y tags
3. ✅ **Transición a Staging** para pruebas pre-producción
4. ✅ **Transición a Production** después de aprobación
5. ✅ **Carga del modelo** desde Registry
6. ✅ **Consulta de versiones** e historial

**Beneficios del Model Registry:**

* **Centralización**: Un solo lugar para todos los modelos
* **Versionado**: Historial completo de cambios
* **Gobernanza**: Control de acceso y aprobaciones
* **Trazabilidad**: Desde datos hasta predicciones
* **Colaboración**: Equipo completo puede acceder y gestionar modelos
* **Automatización**: Integración con CI/CD pipelines

**Próximos pasos:**

* Configurar monitoreo de modelo en producción
* Implementar CI/CD para despliegue automatizado
* Establecer proceso de reentrenamiento
* Configurar alertas de degradación de rendimiento

## 🚀 Despliegue de Modelos y CI/CD para MLOps

Esta sección cubre las mejores prácticas para desplegar modelos ML en producción y establecer pipelines de CI/CD.

### 📊 Opciones de Despliegue en Databricks

**1. Inferencia Batch (Por Lotes)**
* Procesar grandes volúmenes de datos periódicamente
* Ideal para: Clasificación diaria/semanal de contribuyentes
* Implementación: Jobs programados en Databricks
* Ventajas: Eficiente para grandes volúmenes, menor costo

**2. Inferencia en Tiempo Real**
* Predicciones instantáneas vía API REST
* Ideal para: Evaluación de riesgo durante declaraciones
* Implementación: Model Serving endpoints
* Ventajas: Baja latencia, integración con aplicaciones

**3. Inferencia Streaming**
* Procesamiento continuo de datos en tiempo real
* Ideal para: Monitoreo continuo de transacciones
* Implementación: Structured Streaming + MLflow
* Ventajas: Detección inmediata de anomalías

### 🔄 CI/CD para Machine Learning

**Continuous Integration (CI)**:
* Pruebas automáticas de código
* Validación de calidad de datos
* Pruebas unitarias de funciones de preprocesamiento
* Validación de esquemas de datos

**Continuous Deployment (CD)**:
* Despliegue automatizado de modelos
* Promoción automática entre ambientes (Dev → Staging → Prod)
* Rollback automático en caso de fallos
* Monitoreo post-despliegue

### 📦 Databricks Asset Bundles (DABs)

Databricks Asset Bundles es el framework recomendado para CI/CD de proyectos ML en Databricks.

**¿Qué son los DABs?**

Archivos de configuración YAML que definen:
* Jobs de entrenamiento y batch inference
* Pipelines de datos (DLT)
* Notebooks y código fuente
* Configuraciones de ambientes (dev/staging/prod)
* Permisos y recursos

**Ventajas:**

* **Infraestructura como Código**: Todo versionado en Git
* **Ambientes Múltiples**: Dev, Staging, Production con una sola configuración
* **Despliegue Automatizado**: `databricks bundle deploy`
* **Validación**: Verificación de configuración antes de despliegue
* **Rollback Fácil**: Volver a versiones anteriores

**Estructura de proyecto recomendada:**

```
proyecto-riesgo-fiscal/
├── databricks.yml          # Configuración principal del bundle
├── resources/
│   ├── jobs.yml            # Definición de jobs
│   └── model_serving.yml   # Configuración de serving
├── src/
│   ├── training.py         # Script de entrenamiento
│   ├── inference.py        # Script de inferencia
│   └── preprocessing.py    # Funciones de preprocesamiento
├── notebooks/
│   └── exploratory.py      # Notebooks de análisis
├── tests/
│   └── test_preprocessing.py
└── requirements.txt        # Dependencias Python
```

### 🎯 Mejores Prácticas de MLOps

#### 1. Control de Versiones
* ✅ Todo el código en Git (notebooks como .py)
* ✅ Versionado de datos con Delta Lake
* ✅ Versionado de modelos con MLflow
* ✅ Versionado de configuraciones (DABs)

#### 2. Testing
* ✅ Tests unitarios para funciones de preprocesamiento
* ✅ Tests de integración para pipelines completos
* ✅ Validación de esquemas de datos
* ✅ Tests de rendimiento del modelo (métricas mínimas)
* ✅ Tests de regresión (comparar con versión anterior)

#### 3. Monitoreo
* ✅ Métricas de rendimiento del modelo
* ✅ Data drift detection (cambios en distribución de datos)
* ✅ Model drift detection (degradación de rendimiento)
* ✅ Latencia y throughput de inferencia
* ✅ Alertas automáticas

#### 4. Documentación
* ✅ README con instrucciones de setup
* ✅ Documentación de API del modelo
* ✅ Descripciones en MLflow Registry
* ✅ Changelog de versiones
* ✅ Runbooks para operaciones

#### 5. Seguridad y Gobernanza
* ✅ Control de acceso basado en roles (RBAC)
* ✅ Auditoría de cambios
* ✅ Encriptación de datos sensibles
* ✅ Cumplimiento normativo (GDPR, etc.)
* ✅ Proceso de aprobación para producción

#### 6. Reentrenamiento
* ✅ Calendario de reentrenamiento (mensual/trimestral)
* ✅ Triggers automáticos por degradación
* ✅ Validación antes de reemplazar modelo en producción
* ✅ A/B testing de nuevas versiones
* ✅ Rollback automático si falla

## 🎉 Conclusión

### 📊 Resumen del Laboratorio

En este notebook hemos cubierto el ciclo de vida completo de un modelo de Machine Learning para clasificación de riesgo fiscal:

1. ✅ **Generación de Datos Sintéticos** - Dataset realista de contribuyentes
2. ✅ **Análisis Exploratorio (EDA)** - Comprensión profunda de los datos
3. ✅ **Preprocesamiento** - Pipeline robusto con sklearn
4. ✅ **Entrenamiento de Modelos** - Múltiples algoritmos con MLflow tracking
5. ✅ **Optimización** - Hyperparameter tuning con Optuna
6. ✅ **Evaluación** - Métricas completas y visualizaciones
7. ✅ **Registro en MLflow** - Model Registry con versionado
8. ✅ **Despliegue y CI/CD** - Databricks Asset Bundles

### 📚 Recursos Adicionales

**Documentación:**
* [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
* [Databricks Asset Bundles](https://docs.databricks.com/dev-tools/bundles/index.html)
* [MLOps Best Practices](https://ml-ops.org/)

**Capacitación:**
* Databricks Academy - MLOps courses
* MLflow tutorials y webinars
---

**🌟 ¡Felicidades por completar este laboratorio de MLOps con Databricks!**